## 1. Importar librerías

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/b/dataset_macro_gastos.csv")

X_nombre = df['nombre_tienda'].astype(str).values
X_contexto = df['contexto_micro'].astype(str).values

label_encoder_macro = LabelEncoder()
label_encoder_micro = LabelEncoder()

y_macro = label_encoder_macro.fit_transform(df['categoria_macro'])
y_micro = label_encoder_micro.fit_transform(df['contexto_micro'])

num_clases_macro = len(label_encoder_macro.classes_)
num_clases_micro = len(label_encoder_micro.classes_)

X_nom_train, X_nom_test, X_ctx_train, X_ctx_test, y_mac_train, y_mac_test, y_mic_train, y_mic_test = train_test_split(
    X_nombre, X_contexto, y_macro, y_micro, test_size=0.2, random_state=42
)

In [ ]:
max_tokens = 5000
sequence_length = 5

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.adapt(np.concatenate((X_nom_train, X_ctx_train)))

In [ ]:
def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    attn_output = layers.Dropout(dropout_rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)

    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)
    return layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

embed_dim = 64
num_heads = 2
ff_dim = 64

input_nombre = layers.Input(shape=(1,), dtype=tf.string, name='input_nombre')
input_contexto = layers.Input(shape=(1,), dtype=tf.string, name='input_contexto')

emb_nombre = layers.Embedding(input_dim=max_tokens, output_dim=embed_dim)(vectorize_layer(input_nombre))
emb_contexto = layers.Embedding(input_dim=max_tokens, output_dim=embed_dim)(vectorize_layer(input_contexto))

trans_nombre = transformer_encoder(emb_nombre, embed_dim, num_heads, ff_dim)
trans_contexto = transformer_encoder(emb_contexto, embed_dim, num_heads, ff_dim)

pool_nombre = layers.GlobalAveragePooling1D()(trans_nombre)
pool_contexto = layers.GlobalAveragePooling1D()(trans_contexto)

context_fusion = layers.Concatenate(name='context_fusion')([pool_nombre, pool_contexto])
context_fusion = layers.Dense(128, activation='relu')(context_fusion)
context_fusion = layers.Dropout(0.2)(context_fusion)

output_macro = layers.Dense(num_clases_macro, activation='softmax', name='salida_macro')(context_fusion)
output_micro = layers.Dense(num_clases_micro, activation='softmax', name='salida_micro')(context_fusion)

model = Model(inputs=[input_nombre, input_contexto], outputs=[output_macro, output_micro])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'salida_macro': 'sparse_categorical_crossentropy', 'salida_micro': 'sparse_categorical_crossentropy'},
    metrics={'salida_macro': 'accuracy', 'salida_micro': 'accuracy'}
)

model.summary()

In [ ]:
history = model.fit(
    {'input_nombre': X_nom_train, 'input_contexto': X_ctx_train},
    {'salida_macro': y_mac_train, 'salida_micro': y_mic_train},
    validation_data=(
        {'input_nombre': X_nom_test, 'input_contexto': X_ctx_test},
        {'salida_macro': y_mac_test, 'salida_micro': y_mic_test}
    ),
    epochs=10,
    batch_size=32
)

In [ ]:
model.save("modelo_clasificador_transacciones.keras")
print("Modelo guardado exitosamente como .keras")

config_vectorizador = {
    'vocabulario': vectorize_layer.get_vocabulary()
}

artefactos = {
    'label_encoder_macro': label_encoder_macro,
    'label_encoder_micro': label_encoder_micro,
    'config_vectorizador': config_vectorizador
}

with open('artefactos_clasificacion.pkl', 'wb') as f:
    pickle.dump(artefactos, f)